# 04 - Analyse du marché : niches, comparables et prix

Fonctions dans `src/analysis.py`. Revenus = revenu net estimé, scénario moyen.

In [1]:
import sys
sys.path.append('..')

import pandas as pd
from src.cleaning import load_clean
from src.estimation import add_estimates
from src.analysis import (PROFITABLE_THRESHOLD, count_tags_loop, explode_tags,
                          niche_table, tag_pair_table)

pd.options.display.float_format = '{:,.2f}'.format
df = add_estimates(load_clean('../data/games_clean.csv'))
len(df)

71352

# Étape 7 - Les niches

## 1. Compter les tags à la main (boucle + dictionnaire)

In [2]:
counts = count_tags_loop(df)
top10_loop = sorted(counts.items(), key=lambda pair: pair[1], reverse=True)[:10]
top10_loop

[('Singleplayer', 44588),
 ('Indie', 41885),
 ('Casual', 32029),
 ('Action', 31229),
 ('Adventure', 30633),
 ('2D', 24054),
 ('3D', 16274),
 ('Simulation', 15645),
 ('Strategy', 15116),
 ('Puzzle', 14593)]

## 2. Le même comptage avec pandas

In [3]:
counts_pandas = explode_tags(df)['tag'].value_counts()
counts_pandas.head(10)

tag
Singleplayer    44588
Indie           41885
Casual          32029
Action          31229
Adventure       30633
2D              24054
3D              16274
Simulation      15645
Strategy        15116
Puzzle          14593
Name: count, dtype: int64

In [4]:
same = pd.Series(counts).sort_index().equals(counts_pandas.sort_index())
print('Comptages identiques :', same, '|', len(counts), 'tags différents')

Comptages identiques : True | 451 tags différents


## 3. Médiane ou moyenne ?

In [5]:
indie = explode_tags(df).query('tag == "Indie"')['revenue_mid']
print('Indie - revenu moyen  :', round(indie.mean()), '$')
print('Indie - revenu médian :', round(indie.median()), '$')

Indie - revenu moyen  : 278376 $
Indie - revenu médian : 1509 $


La moyenne est des dizaines de fois plus haute que la médiane : quelques succès à plusieurs millions l'écrasent. La médiane décrit le jeu « typique », c'est elle qu'on utilise.

## 4. Le tableau des niches (au moins 50 jeux par tag, rentable = plus de 100 000 $)

In [6]:
niches = niche_table(df)
print(len(niches), 'tags | minimum de jeux :', niches['n_games'].min())
niches.head(15)

418 tags | minimum de jeux : 50


,n_games,median_revenue,mean_revenue,profitable_share,recent_releases,previous_releases,growth
tag,,,,,,,
Games Workshop,62,"573,560.66","10,974,927.65",0.60,10,8,0.25
Kickstarter,133,"102,773.29","2,067,249.97",0.51,5,8,-0.38
Mod,50,"92,391.04","7,435,789.68",0.46,7,8,-0.12
Dwarf,76,"79,444.17","8,873,478.91",0.47,23,7,2.29
Cult Classic,173,"77,307.05","1,024,908.25",0.45,9,12,-0.25
Crowdfunded,81,"56,980.66","1,501,363.39",0.38,6,7,-0.14
Moddable,726,"49,310.38","5,649,913.68",0.43,212,164,0.29
Great Soundtrack,3167,"45,623.65","5,462,273.06",0.41,141,226,-0.38
Remake,332,"38,289.98","4,558,786.54",0.41,62,53,0.17


**Attention** : les premiers tags (Games Workshop, Kickstarter, Cult Classic, Great Soundtrack...) ne sont pas des concepts de jeu.
Ce sont des étiquettes de licence, de financement, ou données **après** le succès (un jeu devient « Cult Classic » parce qu'il a marché).
Un studio ne peut pas les choisir : on les écarte pour choisir des niches.

In [7]:
# Tags qui décrivent un financement, une licence ou un succès passé, pas un genre.
META_TAGS = ['Games Workshop', 'Kickstarter', 'Crowdfunded', 'Mod', 'Moddable', 'Cult Classic',
             'Great Soundtrack', 'Classic', 'Remake', 'Sequel', 'Epic', 'Masterpiece',
             'Mature', 'Hentai', 'NSFW', 'Sexual Content', 'Nudity', 'Dwarf']
genres = niches.drop(index=[t for t in META_TAGS if t in niches.index])

## 5. Niches prometteuses : peu de jeux, revenu médian élevé, bonne part de jeux rentables, et qui ne se remplissent pas trop vite

In [8]:
promising = genres[(genres['n_games'] <= 1000)
                   & (genres['profitable_share'] >= 0.30)
                   & (genres['growth'] < 1)]
promising.head(8)

,n_games,median_revenue,mean_revenue,profitable_share,recent_releases,previous_releases,growth
tag,,,,,,,
Villain Protagonist,204,"35,071.88","807,251.88",0.40,55,42,0.31
Real-Time with Pause,240,"33,072.35","4,788,802.77",0.40,46,45,0.02
Open World Survival Craft,497,"28,713.64","9,104,588.38",0.37,152,119,0.28
4X,349,"18,121.32","2,944,443.54",0.35,111,77,0.44
Heist,232,"14,065.38","3,515,057.79",0.34,69,64,0.08
Trains,285,"12,349.26","1,219,461.22",0.32,97,54,0.80
Motorbike,156,"12,227.64","1,043,492.77",0.35,42,43,-0.02
MMORPG,169,"12,022.10","7,924,067.61",0.31,57,34,0.68


**5 niches prometteuses** :
1. **Real-Time with Pause** (240 jeux) : revenu médian ~33 000 $, 40 % de jeux rentables, sorties stables : un public fidèle et peu de nouveaux concurrents.
2. **Villain Protagonist** (204 jeux) : ~35 000 $ de médiane, 40 % rentables : un angle narratif original qui se démarque.
3. **Open World Survival Craft** (497 jeux) : ~29 000 $ de médiane, 37 % rentables : un genre très joué (Valheim), encore loin de la saturation.
4. **4X** (349 jeux) : ~18 000 $ de médiane, 35 % rentables : public de stratégie prêt à payer, mais sorties en hausse (+44 %).
5. **Heist** (232 jeux) : ~14 000 $ de médiane, 34 % rentables, croissance faible (+8 %).

## 6. Niches à éviter : beaucoup de jeux et très peu de jeux rentables

In [9]:
to_avoid = genres[genres['n_games'] >= 1000].sort_values('profitable_share').head(8)
to_avoid

,n_games,median_revenue,mean_revenue,profitable_share,recent_releases,previous_releases,growth
tag,,,,,,,
Runner,1479,508.60,"21,591.79",0.03,529,501,0.06
Abstract,1977,904.93,"25,036.25",0.03,627,571,0.10
Clicker,2065,744.66,"22,379.69",0.03,796,669,0.19
Minimalist,5167,677.00,"48,592.75",0.03,1690,1683,0.00
Logic,4030,906.07,"68,211.13",0.03,1453,1330,0.09
Hidden Object,4081,"1,187.87","29,044.90",0.03,1524,1049,0.45
Linear,5683,792.67,"46,286.83",0.04,2444,1886,0.30
Precision Platformer,2108,755.24,"63,116.78",0.04,826,701,0.18


**3 niches à éviter** :
1. **Minimalist** (plus de 5 000 jeux) : revenu médian ~680 $, 3 % de jeux rentables.
2. **Clicker** (plus de 2 000 jeux) : ~740 $ de médiane, 3 % rentables, et +19 % de sorties.
3. **Precision Platformer** (plus de 2 000 jeux) : ~750 $ de médiane, 4 % rentables.

À surveiller : **Roguelike Deckbuilder** a vu ses sorties presque tripler (+170 %) : la niche se remplit très vite depuis le succès de Balatro et Slay the Spire.

In [10]:
niches.loc[['Roguelike Deckbuilder', 'Rogue-like', 'Cozy', 'Deckbuilding']]

,n_games,median_revenue,mean_revenue,profitable_share,recent_releases,previous_releases,growth
tag,,,,,,,
Roguelike Deckbuilder,418,"14,938.37","556,264.91",0.22,208,77,1.70
Rogue-like,4565,"2,833.11","402,769.38",0.15,2094,910,1.30
Cozy,713,"4,610.65","176,629.91",0.18,378,114,2.32
Deckbuilding,1166,"4,599.88","369,319.26",0.17,591,244,1.42


## 7. Bonus : les paires de tags

In [11]:
pairs = tag_pair_table(df)
pairs[pairs['n_games'] <= 1000].head(10)

,n_games,median_revenue,profitable_share
pair,,,
Multiplayer + Story Rich,542,"415,619.13",0.59
Multiplayer + Visual Novel,60,"128,791.21",0.52
Multiplayer + RPG,953,"94,110.66",0.50
Female Protagonist + Multiplayer,309,"94,027.88",0.50
FPS + Visual Novel,140,"80,158.87",0.47
Anime + Multiplayer,450,"71,705.65",0.46
Difficult + Multiplayer,528,"51,183.09",0.45
Anime + Open World,300,"49,733.55",0.46
Female Protagonist + Open World,438,"36,489.76",0.42


In [12]:
# Pourquoi des paires étranges (FPS + Visual Novel) rapportent-elles autant ?
n_tags = df['tags'].str.count(',') + 1
print('Corrélation nombre de tags / nombre d\'avis :', round(n_tags.corr(df['reviews'], method='spearman'), 2))
print(df.assign(n_tags=n_tags).groupby(pd.cut(n_tags, [0, 5, 10, 15, 20]), observed=True)['revenue_mid'].median())

Corrélation nombre de tags / nombre d'avis : 0.2
tags
(0, 5]     1,017.48
(5, 10]    1,341.90
(10, 15]     960.69
(15, 20]   2,469.85
Name: revenue_mid, dtype: float64


**Piège d'interprétation** : sur Steam, ce sont les joueurs qui posent les tags. Le lien entre nombre de tags et popularité est modéré (corrélation 0,2), mais les jeux à plus de 15 tags ont un revenu médian 2 fois plus élevé.
Les paires inattendues (« FPS + Visual Novel », « Multiplayer + Story Rich ») apparaissent donc surtout sur de gros jeux, ce qui gonfle leur revenu médian.
Conclusion : une paire n'est utile que si elle décrit un vrai concept (ex. Roguelike + Deckbuilding) ; le classement brut des paires n'est pas une liste de niches à viser.

In [13]:
pairs.loc[[p for p in ['Card Game + Rogue-like', 'Indie + Rogue-like', 'Cozy + Indie', 'Farming Sim + Indie'] if p in pairs.index]]

,n_games,median_revenue,profitable_share
pair,,,
Indie + Rogue-like,2404,"3,016.44",0.16


# Étape 8 - Jeux comparables et analyse des prix

## 1. Tester `has_all_tags` sur 4 cas écrits à la main

In [14]:
from src.analysis import (has_all_tags, find_comparables, price_table, recommend_price,
                          MIN_COMPARABLES, PRICE_TOLERANCE)

print(has_all_tags('Indie,Rogue-like,Action', ['Indie', 'Roguelike']))  # tous les tags -> True
print(has_all_tags('Indie,Action', ['Indie', 'Roguelike']))             # un tag manquant -> False
print(has_all_tags('Indie,Action', ['indie']))                          # casse différente -> True
print(has_all_tags('Indie,Action', []))                                 # liste vide -> True

True
False
True
True


## 2. Notre fil rouge

Un studio de **2 personnes, 60 000 $ de budget**, hésite entre deux concepts vendus **15 $** :
- **A** : un roguelike (tags Indie + Roguelike) ;
- **B** : un cozy game (tags Indie + Cozy).

In [15]:
CONCEPTS = {
    'A - Roguelike': ['Indie', 'Roguelike'],
    'B - Cozy': ['Indie', 'Cozy'],
    'Open World Survival Craft': ['Open World Survival Craft'],
    'Clicker': ['Clicker'],
}
for name, tags in CONCEPTS.items():
    all_prices = find_comparables(df, tags)
    near_price = find_comparables(df, tags, price=15)
    recent = find_comparables(df, tags, price=15, since_year=2020)
    print(f'{name:28} tags seuls: {len(all_prices):5} | prix 15$ ±50%: {len(near_price):4} | et depuis 2020: {len(recent):4}')
print('Minimum de comparables fiable :', MIN_COMPARABLES)

A - Roguelike                tags seuls:  2404 | prix 15$ ±50%:  894 | et depuis 2020:  625


B - Cozy                     tags seuls:   385 | prix 15$ ±50%:  125 | et depuis 2020:  119


Open World Survival Craft    tags seuls:   497 | prix 15$ ±50%:  284 | et depuis 2020:  216


Clicker                      tags seuls:  2065 | prix 15$ ±50%:  272 | et depuis 2020:  234
Minimum de comparables fiable : 30


Il y a largement plus de 30 comparables pour chaque concept. Le minimum de 30 (constante `MIN_COMPARABLES`) servira dans le simulateur pour prévenir l'utilisateur quand une niche est trop petite.

## 3. Revenu par tranche de prix

In [16]:
roguelike = find_comparables(df, ['Indie', 'Roguelike'])
print(len(roguelike), 'comparables Indie + Roguelike')
price_table(roguelike)

2404 comparables Indie + Roguelike


,n_games,median_revenue,profitable_share
price_band,,,
<5$,1196,754.11,0.02
5-10$,607,"5,475.52",0.10
10-15$,333,"38,684.52",0.35
15-20$,188,"209,891.02",0.60
20-30$,74,"984,828.73",0.82
30$+,6,"83,331.71",0.50


In [17]:
print('Tranche recommandée :', recommend_price(roguelike))

Tranche recommandée : 20-30$


In [18]:
for name, tags in CONCEPTS.items():
    print(f'{name:28} -> {recommend_price(find_comparables(df, tags))}')

A - Roguelike                -> 20-30$


B - Cozy                     -> 15-20$


Open World Survival Craft    -> 30$+


Clicker                      -> 30$+


In [19]:
price_table(find_comparables(df, ['Indie', 'Cozy']))

,n_games,median_revenue,profitable_share
price_band,,,
<5$,224,698.54,0.02
5-10$,73,"8,943.67",0.16
10-15$,40,"24,157.22",0.30
15-20$,38,"111,179.82",0.53
20-30$,9,"729,136.48",0.78
30$+,1,"582,448.48",1.00


**Lecture** : dans la niche Indie + Roguelike, un jeu à 15-20 $ rapporte en médiane environ **280 fois plus** qu'un jeu à moins de 5 $ (≈ 210 000 $ contre ≈ 750 $).
La tranche recommandée change selon la niche : 20-30 $ pour les roguelikes, 15-20 $ pour les cozy games (la tranche 20-30 $ n'a que 9 jeux, en dessous du minimum de 10).

**Prudence** : c'est une corrélation, pas une causalité. Un jeu vendu 25 $ a souvent un plus gros budget de production et plus de contenu qu'un jeu à 3 $ ;
baisser son prix ne fera pas vendre plus. L'outil conseille donc une tranche « où les jeux comparables réussissent », pas un prix magique.
Pour la même raison, les très petites tranches (Clicker à 30 $ et plus) donnent des recommandations fragiles.